In [1]:
import numpy as np

In [2]:
import numpy as np

def butterfly(a, b, twiddle):
    """
    a, b: Liczby zespolone (wejścia)
    twiddle: Liczba zespolona (współczynnik W_N^k)
    """
    # Operacja motylkowa:
    # A = a + b*W
    # B = a - b*W
    temp = b * twiddle
    return a + temp, a - temp

def fft_radix2(x):
    N = len(x)
    if N <= 1: return x
    
    # Podział na parzyste i nieparzyste
    even = fft_radix2(x[0::2])
    odd = fft_radix2(x[1::2])
    
    # Współczynniki Twiddle
    T = [np.exp(-2j * np.pi * k / N) for k in range(N // 2)]
    
    # Łączenie (Butterfly)
    return [even[k] + T[k] * odd[k] for k in range(N // 2)] + \
           [even[k] - T[k] * odd[k] for k in range(N // 2)]

In [ ]:
def fixed_point_butterfly(A_re, A_im, B_re, B_im, W_re, W_im):
    """
    Wszystkie argumenty wejściowe muszą być liczbami całkowitymi (int)
    reprezentującymi ułamki w formacie Q1.15.
    """
    
    # ----------------------------------------------------
    # KROK 1: Mnożenie zespolone (B * W)
    # W FPGA mnożenie dwóch liczb 16-bitowych daje wynik 32-bitowy (format Q2.30)
    # ----------------------------------------------------
    temp_re = (B_re * W_re) - (B_im * W_im)
    temp_im = (B_re * W_im) + (B_im * W_re)
    
    # ----------------------------------------------------
    # KROK 2: Skalowanie (Bit-shift)
    # Przesunięcie bitowe o 15 w prawo ucina "nadmiarowe" bity ułamkowe 
    # i przywraca wynik mnożenia do bazowego formatu Q1.15
    # ----------------------------------------------------
    temp_re = temp_re >> 15
    temp_im = temp_im >> 15
    
    # ----------------------------------------------------
    # KROK 3: Operacja sumowania i odejmowania (Butterfly)
    # ----------------------------------------------------
    out_A_re = A_re + temp_re
    out_A_im = A_im + temp_im
    
    out_B_re = A_re - temp_re
    out_B_im = A_im - temp_im
    
    # W rygorystycznym modelu hardware'owym tutaj następuje ew. obcięcie do 16-bit
    # np. ograniczenie wyników do zakresu [-32768, 32767] (Saturacja)
    
    return out_A_re, out_A_im, out_B_re, out_B_im

In [50]:
z1:complex = 0.1+0.2j
z2:complex = 0.3-0.5j

print(fixed_point_butterfly(z1,z2,16))


[np.int16(8519), np.int16(655)]
